In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"F:\yingxiang2")

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "radiomics_features.csv"
LABEL_PATH = PROJECT_ROOT / "final_patients_egfr.csv"

features_df = pd.read_csv(FEATURE_PATH)
labels_df = pd.read_csv(LABEL_PATH)

print("Features:", features_df.shape)
print("Labels:", labels_df.shape)

display(features_df.head())
display(labels_df.head())

Features: (97, 108)
Labels: (97, 2)


,Case ID,original_shape_Elongation,original_shape_Flatness,original_shape_LeastAxisLength,original_shape_MajorAxisLength,original_shape_Maximum2DDiameterColumn,original_shape_Maximum2DDiameterRow,original_shape_Maximum2DDiameterSlice,original_shape_Maximum3DDiameter,original_shape_MeshVolume,...,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,R01-003,0.720221,0.657195,11.036269,16.792985,15.264338,20.396078,20.248457,21.908902,1436.000000,...,0.015595,0.001915,0.381474,399.314034,0.001424,0.094470,0.004453,3667.838634,0.214884,5.187657
1,R01-005,0.785596,0.671483,18.349358,27.326629,30.413813,28.442925,32.557641,34.234486,7093.583333,...,0.039476,0.001121,0.196453,180.248868,0.000467,0.495823,0.000862,1688.359596,0.052741,1.630667
2,R01-006,0.827038,0.611589,14.872624,24.317993,27.856777,27.018512,27.294688,31.208973,4185.458333,...,0.063193,0.004919,0.279960,150.481212,0.002754,0.319900,0.002256,1444.603853,0.232923,1.882577
3,R01-008,0.962727,0.606142,15.220509,25.110479,30.886890,28.442925,30.413813,31.906112,6136.166667,...,0.047396,0.005656,0.318292,153.986034,0.002414,0.484510,0.002295,2593.958310,0.211343,2.257230
4,R01-012,0.825965,0.726582,47.295007,65.092457,71.700767,73.246160,76.850504,78.057671,110072.875000,...,0.072685,0.004176,0.263579,158.757944,0.001111,10.450988,0.000102,2952.279138,0.171360,0.112334


,Case ID,EGFR mutation status
0,R01-003,Mutant
1,R01-005,Wildtype
2,R01-006,Wildtype
3,R01-008,Wildtype
4,R01-012,Wildtype


In [2]:
model_df = features_df.merge(
    labels_df,
    on="Case ID",
    how="inner",
    validate="one_to_one"
)

print("Merged shape:", model_df.shape)

print("\nEGFR distribution:")
print(model_df["EGFR mutation status"].value_counts(dropna=False))

print("\nDuplicate Case IDs:")
print(model_df["Case ID"].duplicated().sum())

print("\nMissing values:")
print(model_df.isna().sum().sum())

Merged shape: (97, 109)

EGFR distribution:
EGFR mutation status
Wildtype    74
Mutant      23
Name: count, dtype: int64

Duplicate Case IDs:
0

Missing values:
0


In [3]:
# 107个影像组学特征
feature_cols = [
    c for c in model_df.columns
    if c.startswith("original_")
]

X = model_df[feature_cols].copy()

# Mutant = 1, Wildtype = 0
y = (
    model_df["EGFR mutation status"]
    .map({
        "Wildtype": 0,
        "Mutant": 1
    })
    .astype(int)
)

case_ids = model_df["Case ID"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts().sort_index())

print("\nFeature count:", len(feature_cols))

X shape: (97, 107)
y shape: (97,)

Class distribution:
EGFR mutation status
0    74
1    23
Name: count, dtype: int64

Feature count: 107


In [5]:
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE = 2026

# 外层：只用于最终泛化性能评估
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# 内层：只用于特征选择和调参
inner_cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Outer CV:", outer_cv)
print("Inner CV:", inner_cv)

Outer CV: StratifiedKFold(n_splits=5, random_state=2026, shuffle=True)
Inner CV: StratifiedKFold(n_splits=4, random_state=2026, shuffle=True)


In [6]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts().sort_index())

X shape: (97, 107)
y shape: (97,)
EGFR mutation status
0    74
1    23
Name: count, dtype: int64


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=mutual_info_classif)),
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

param_grid = {
    "selector__k": [5, 10, 15, 20],
    "model__C": [0.01, 0.1, 1, 10]
}

print(pipe)

Pipeline(steps=[('scaler', StandardScaler()),
                ('selector',
                 SelectKBest(score_func=<function mutual_info_classif at 0x0000027B7E07EA20>)),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=2026, solver='liblinear'))])


In [8]:
oof_pred = np.zeros(len(X), dtype=float)
fold_rows = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        refit=True
    )

    grid.fit(X_train, y_train)

    pred = grid.predict_proba(X_test)[:, 1]
    oof_pred[test_idx] = pred

    fold_auc = roc_auc_score(y_test, pred)
    fold_ap = average_precision_score(y_test, pred)

    fold_rows.append({
        "fold": fold,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "n_mutant_test": int(y_test.sum()),
        "best_k": grid.best_params_["selector__k"],
        "best_C": grid.best_params_["model__C"],
        "inner_best_auc": grid.best_score_,
        "outer_auc": fold_auc,
        "outer_pr_auc": fold_ap,
    })

    print(
        f"Fold {fold}: "
        f"AUC={fold_auc:.3f}, "
        f"PR-AUC={fold_ap:.3f}, "
        f"best_k={grid.best_params_['selector__k']}, "
        f"best_C={grid.best_params_['model__C']}"
    )

Fold 1: AUC=0.173, PR-AUC=0.184, best_k=20, best_C=10
Fold 2: AUC=0.533, PR-AUC=0.317, best_k=15, best_C=10
Fold 3: AUC=0.371, PR-AUC=0.242, best_k=20, best_C=0.01
Fold 4: AUC=0.500, PR-AUC=0.243, best_k=5, best_C=10
Fold 5: AUC=0.383, PR-AUC=0.202, best_k=15, best_C=1


In [9]:
fold_df = pd.DataFrame(fold_rows)

overall_auc = roc_auc_score(y, oof_pred)
overall_pr_auc = average_precision_score(y, oof_pred)

print("\nOOF ROC-AUC:", round(overall_auc, 3))
print("OOF PR-AUC:", round(overall_pr_auc, 3))

display(fold_df)


OOF ROC-AUC: 0.391
OOF PR-AUC: 0.197


,fold,n_train,n_test,n_mutant_test,best_k,best_C,inner_best_auc,outer_auc,outer_pr_auc
0,1,77,20,5,20,10.00,0.675714,0.173333,0.183857
1,2,77,20,5,15,10.00,0.534643,0.533333,0.317222
2,3,78,19,5,20,0.01,0.629167,0.371429,0.241940
3,4,78,19,4,5,10.00,0.611667,0.500000,0.243056
4,5,78,19,4,15,1.00,0.568452,0.383333,0.202492


In [10]:
print("y classes:", sorted(y.unique()))
print("Mutant count:", int(y.sum()))

print(
    "Example model classes:",
    grid.best_estimator_.named_steps["model"].classes_
)

print(
    "Mean predicted probability:",
    oof_pred.mean()
)

print(
    "Mean pred for Mutant:",
    oof_pred[y.values == 1].mean()
)

print(
    "Mean pred for Wildtype:",
    oof_pred[y.values == 0].mean()
)

y classes: [0, 1]
Mutant count: 23
Example model classes: [0 1]
Mean predicted probability: 0.4400874703354336
Mean pred for Mutant: 0.377601121115048
Mean pred for Wildtype: 0.45950890320122895


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

pipe_l2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

param_grid_l2 = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

oof_pred_l2 = np.zeros(len(X), dtype=float)
fold_rows_l2 = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    grid_l2 = GridSearchCV(
        estimator=pipe_l2,
        param_grid=param_grid_l2,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        refit=True
    )

    grid_l2.fit(X_train, y_train)

    pred = grid_l2.predict_proba(X_test)[:, 1]
    oof_pred_l2[test_idx] = pred

    fold_auc = roc_auc_score(y_test, pred)
    fold_ap = average_precision_score(y_test, pred)

    fold_rows_l2.append({
        "fold": fold,
        "best_C": grid_l2.best_params_["model__C"],
        "inner_best_auc": grid_l2.best_score_,
        "outer_auc": fold_auc,
        "outer_pr_auc": fold_ap,
    })

    print(
        f"Fold {fold}: "
        f"AUC={fold_auc:.3f}, "
        f"PR-AUC={fold_ap:.3f}, "
        f"C={grid_l2.best_params_['model__C']}"
    )

fold_l2_df = pd.DataFrame(fold_rows_l2)

print("\nL2 Logistic OOF ROC-AUC:",
      round(roc_auc_score(y, oof_pred_l2), 3))

print("L2 Logistic OOF PR-AUC:",
      round(average_precision_score(y, oof_pred_l2), 3))

display(fold_l2_df)

C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Fold 1: AUC=0.307, PR-AUC=0.237, C=100
Fold 2: AUC=0.627, PR-AUC=0.429, C=1


C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Fold 3: AUC=0.857, PR-AUC=0.679, C=100
Fold 4: AUC=0.617, PR-AUC=0.499, C=100
Fold 5: AUC=0.633, PR-AUC=0.365, C=100

L2 Logistic OOF ROC-AUC: 0.573
L2 Logistic OOF PR-AUC: 0.336


C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,fold,best_C,inner_best_auc,outer_auc,outer_pr_auc
0,1,100,0.738690,0.306667,0.237126
1,2,1,0.556786,0.626667,0.428695
2,3,100,0.604167,0.857143,0.678730
3,4,100,0.624762,0.616667,0.499359
4,5,100,0.647381,0.633333,0.364583


In [12]:
print("Mean pred Mutant:",
      oof_pred_l2[y.values == 1].mean())

print("Mean pred Wildtype:",
      oof_pred_l2[y.values == 0].mean())

Mean pred Mutant: 0.4351940935549327
Mean pred Wildtype: 0.30833211678648825


In [13]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

repeat_results = []

N_REPEATS = 20

for repeat in range(N_REPEATS):

    seed = 2026 + repeat

    outer_cv_rep = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    inner_cv_rep = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )

    oof_pred_rep = np.zeros(len(X), dtype=float)

    for train_idx, test_idx in outer_cv_rep.split(X, y):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        grid = GridSearchCV(
            estimator=pipe_l2,
            param_grid=param_grid_l2,
            scoring="roc_auc",
            cv=inner_cv_rep,
            n_jobs=-1,
            refit=True
        )

        grid.fit(X_train, y_train)

        oof_pred_rep[test_idx] = (
            grid.predict_proba(X_test)[:, 1]
        )

    auc = roc_auc_score(y, oof_pred_rep)
    pr_auc = average_precision_score(y, oof_pred_rep)

    repeat_results.append({
        "repeat": repeat + 1,
        "seed": seed,
        "roc_auc": auc,
        "pr_auc": pr_auc
    })

    print(
        f"Repeat {repeat + 1:02d}: "
        f"AUC={auc:.3f}, PR-AUC={pr_auc:.3f}"
    )

repeat_df = pd.DataFrame(repeat_results)

print("\nROC-AUC summary:")
print(repeat_df["roc_auc"].describe())

print("\nPR-AUC summary:")
print(repeat_df["pr_auc"].describe())

C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.cond

Repeat 01: AUC=0.573, PR-AUC=0.336


C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.cond

Repeat 02: AUC=0.533, PR-AUC=0.268


C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.cond

...

C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.cond

Repeat 19: AUC=0.605, PR-AUC=0.309


C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Repeat 20: AUC=0.613, PR-AUC=0.385

ROC-AUC summary:
count    20.000000
mean      0.627850
std       0.048885
min       0.533490
25%       0.600323
50%       0.618978
75%       0.660987
max       0.742068
Name: roc_auc, dtype: float64

PR-AUC summary:
count    20.000000
mean      0.352572
std       0.047831
min       0.268126
25%       0.325034
50%       0.356127
75%       0.388365
max       0.444531
Name: pr_auc, dtype: float64


C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Administrator\.conda\envs\lung2\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [14]:
PROTOCOL_PATH = (
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "ct_acquisition_parameters_qc.csv"
)

protocol_df = pd.read_csv(PROTOCOL_PATH)

print("Protocol table shape:", protocol_df.shape)
print("\nColumns:")
for c in protocol_df.columns:
    print(c)

display(protocol_df.head())

Protocol table shape: (97, 13)

Columns:
Case ID
SeriesDescription
SliceThickness
PixelSpacing_X
PixelSpacing_Y
ReconstructionKernel
KVP
Manufacturer
ScannerModel
ContrastAgent
NumberOfSlices
ReconstructionKernel_clean
EGFR mutation status


,Case ID,SeriesDescription,SliceThickness,PixelSpacing_X,PixelSpacing_Y,ReconstructionKernel,KVP,Manufacturer,ScannerModel,ContrastAgent,NumberOfSlices,ReconstructionKernel_clean,EGFR mutation status
0,R01-003,THIN LUNG WINDOW,1.250,0.937500,0.937500,LUNG,120.0,GE MEDICAL SYSTEMS,Discovery CT750 HD,WATER,288,LUNG,Mutant
1,R01-005,Thorax 1.0 B45f,1.000,0.654297,0.654297,B45f,120.0,NaN,NaN,NaN,329,B45f,Wildtype
2,R01-006,.625 mm Chest,0.625,0.976562,0.976562,STANDARD,120.0,GE MEDICAL SYSTEMS,Discovery CT750 HD,WATER,567,STANDARD,Wildtype
3,R01-008,LUNG,3.000,0.750000,0.750000,YB,120.0,NaN,NaN,NaN,199,YB,Wildtype
4,R01-012,Thorax 1.0 B45f,1.000,0.699219,0.699219,B45f,120.0,NaN,NaN,NaN,373,B45f,Wildtype


In [15]:
REPEAT_OUTPUT = (
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_l2_logistic.csv"
)

repeat_df.to_csv(REPEAT_OUTPUT, index=False)

print("Saved:", REPEAT_OUTPUT)
print("Exists:", REPEAT_OUTPUT.exists())

Saved: F:\yingxiang2\results\metrics\repeated_nested_cv_l2_logistic.csv
Exists: True


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

# -----------------------------
# protocol-only predictors
# -----------------------------
protocol_numeric = [
    "SliceThickness",
    "PixelSpacing_X",
    "PixelSpacing_Y",
    "KVP",
    "NumberOfSlices",
]

protocol_categorical = [
    "ReconstructionKernel_clean",
    "Manufacturer",
    "ScannerModel",
    "ContrastAgent",
]

protocol_features = protocol_numeric + protocol_categorical

protocol_model_df = protocol_df[
    ["Case ID", "EGFR mutation status"] + protocol_features
].copy()

print("Protocol model shape:", protocol_model_df.shape)
print("\nMissing values by column:")
print(protocol_model_df[protocol_features].isna().sum())

Protocol model shape: (97, 11)

Missing values by column:
SliceThickness                 0
PixelSpacing_X                 0
PixelSpacing_Y                 0
KVP                            0
NumberOfSlices                 0
ReconstructionKernel_clean     0
Manufacturer                   4
ScannerModel                  42
ContrastAgent                 39
dtype: int64


In [17]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, protocol_numeric),
    ("cat", categorical_pipe, protocol_categorical),
])

protocol_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000,
        random_state=2026
    ))
])

protocol_param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

In [18]:
X_protocol = protocol_model_df[protocol_features].copy()

y_protocol = (
    protocol_model_df["EGFR mutation status"]
    .map({
        "Wildtype": 0,
        "Mutant": 1
    })
    .astype(int)
)

print("X_protocol shape:", X_protocol.shape)
print(y_protocol.value_counts().sort_index())

X_protocol shape: (97, 9)
EGFR mutation status
0    74
1    23
Name: count, dtype: int64


In [19]:
protocol_repeat_results = []

N_REPEATS = 20

for repeat in range(N_REPEATS):

    seed = 2026 + repeat

    outer_cv_rep = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    inner_cv_rep = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )

    oof_pred_protocol = np.zeros(len(X_protocol), dtype=float)

    for train_idx, test_idx in outer_cv_rep.split(X_protocol, y_protocol):

        X_train = X_protocol.iloc[train_idx]
        X_test = X_protocol.iloc[test_idx]

        y_train = y_protocol.iloc[train_idx]
        y_test = y_protocol.iloc[test_idx]

        grid_protocol = GridSearchCV(
            estimator=protocol_pipe,
            param_grid=protocol_param_grid,
            scoring="roc_auc",
            cv=inner_cv_rep,
            n_jobs=-1,
            refit=True
        )

        grid_protocol.fit(X_train, y_train)

        oof_pred_protocol[test_idx] = (
            grid_protocol.predict_proba(X_test)[:, 1]
        )

    auc = roc_auc_score(
        y_protocol,
        oof_pred_protocol
    )

    pr_auc = average_precision_score(
        y_protocol,
        oof_pred_protocol
    )

    protocol_repeat_results.append({
        "repeat": repeat + 1,
        "seed": seed,
        "roc_auc": auc,
        "pr_auc": pr_auc
    })

    print(
        f"Repeat {repeat + 1:02d}: "
        f"AUC={auc:.3f}, "
        f"PR-AUC={pr_auc:.3f}"
    )

protocol_repeat_df = pd.DataFrame(protocol_repeat_results)

print("\nProtocol-only ROC-AUC:")
print(protocol_repeat_df["roc_auc"].describe())

print("\nProtocol-only PR-AUC:")
print(protocol_repeat_df["pr_auc"].describe())

Repeat 01: AUC=0.463, PR-AUC=0.281
Repeat 02: AUC=0.421, PR-AUC=0.244
Repeat 03: AUC=0.538, PR-AUC=0.289
Repeat 04: AUC=0.535, PR-AUC=0.297
Repeat 05: AUC=0.602, PR-AUC=0.376
Repeat 06: AUC=0.630, PR-AUC=0.408
Repeat 07: AUC=0.566, PR-AUC=0.303
Repeat 08: AUC=0.568, PR-AUC=0.325
Repeat 09: AUC=0.529, PR-AUC=0.336
Repeat 10: AUC=0.585, PR-AUC=0.368
Repeat 11: AUC=0.514, PR-AUC=0.274
Repeat 12: AUC=0.540, PR-AUC=0.316
Repeat 13: AUC=0.539, PR-AUC=0.319
Repeat 14: AUC=0.576, PR-AUC=0.386
Repeat 15: AUC=0.526, PR-AUC=0.284
Repeat 16: AUC=0.489, PR-AUC=0.260
Repeat 17: AUC=0.526, PR-AUC=0.327
Repeat 18: AUC=0.590, PR-AUC=0.326
Repeat 19: AUC=0.498, PR-AUC=0.271
Repeat 20: AUC=0.616, PR-AUC=0.337

Protocol-only ROC-AUC:
count    20.000000
mean      0.542626
std       0.051531
min       0.421269
25%       0.522767
50%       0.538484
75%       0.578437
max       0.629847
Name: roc_auc, dtype: float64

Protocol-only PR-AUC:
count    20.000000
mean      0.316293
std       0.043723
min       0.24

In [20]:
protocol_repeat_df.to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_protocol_only.csv",
    index=False
)

In [21]:
comparison_df = repeat_df.merge(
    protocol_repeat_df,
    on=["repeat", "seed"],
    suffixes=("_radiomics", "_protocol")
)

comparison_df["auc_diff"] = (
    comparison_df["roc_auc_radiomics"]
    - comparison_df["roc_auc_protocol"]
)

comparison_df["pr_auc_diff"] = (
    comparison_df["pr_auc_radiomics"]
    - comparison_df["pr_auc_protocol"]
)

print("AUC difference summary:")
print(comparison_df["auc_diff"].describe())

print("\nPR-AUC difference summary:")
print(comparison_df["pr_auc_diff"].describe())

print("\nRadiomics better AUC in:",
      (comparison_df["auc_diff"] > 0).sum(),
      "of", len(comparison_df), "repeats")

AUC difference summary:
count    20.000000
mean      0.085223
std       0.049666
min      -0.003525
25%       0.059342
50%       0.076968
75%       0.113543
max       0.202703
Name: auc_diff, dtype: float64

PR-AUC difference summary:
count    20.000000
mean      0.036279
std       0.048169
min      -0.050557
25%       0.006894
50%       0.033671
75%       0.067392
max       0.125884
Name: pr_auc_diff, dtype: float64

Radiomics better AUC in: 19 of 20 repeats


In [22]:
comparison_df.to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "radiomics_vs_protocol_comparison.csv",
    index=False
)

In [23]:
print(labels_df.columns.tolist())
display(labels_df.head())

['Case ID', 'EGFR mutation status']


,Case ID,EGFR mutation status
0,R01-003,Mutant
1,R01-005,Wildtype
2,R01-006,Wildtype
3,R01-008,Wildtype
4,R01-012,Wildtype


In [24]:
RAW_LABEL_PATH = (
    PROJECT_ROOT
    / "NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv"
)

raw_labels = pd.read_csv(RAW_LABEL_PATH)

print("Shape:", raw_labels.shape)

print("\nColumns:")
for c in raw_labels.columns:
    print(c)

display(raw_labels.head())

Shape: (211, 40)

Columns:
Case ID
Patient affiliation
Age at Histological Diagnosis
Weight (lbs)
Gender
Ethnicity
Smoking status
Pack Years
Quit Smoking Year
%GG
Tumor Location (choice=RUL)
Tumor Location (choice=RML)
Tumor Location (choice=RLL)
Tumor Location (choice=LUL)
Tumor Location (choice=LLL)
Tumor Location (choice=L Lingula)
Tumor Location (choice=Unknown)
Histology 
Pathological T stage
Pathological N stage
Pathological M stage
Histopathological Grade
Lymphovascular invasion
Pleural invasion (elastic, visceral, or parietal)
EGFR mutation status
KRAS mutation status
ALK translocation status
Adjuvant Treatment
Chemotherapy
Radiation
Recurrence
Recurrence Location
Date of Recurrence
Date of Last Known Alive
Survival Status
Date of Death
Time to Death (days)
CT Date
Days between CT and surgery
PET Date


,Case ID,Patient affiliation,Age at Histological Diagnosis,Weight (lbs),Gender,Ethnicity,Smoking status,Pack Years,Quit Smoking Year,%GG,...,Recurrence,Recurrence Location,Date of Recurrence,Date of Last Known Alive,Survival Status,Date of Death,Time to Death (days),CT Date,Days between CT and surgery,PET Date
0,AMC-001,Stanford,34,Not Collected,Male,Not Recorded In Database,Nonsmoker,NaN,NaN,Not Assessed,...,yes,distant,10/7/1994,1/7/1997,Dead,1/7/1997,872.0,8/10/1994,9,Not Collected
1,AMC-002,Stanford,33,Not Collected,Female,Not Recorded In Database,Nonsmoker,NaN,NaN,Not Assessed,...,no,NaN,NaN,3/20/1992,Alive,NaN,NaN,2/19/1992,3,Not Collected
2,AMC-003,Stanford,69,Not Collected,Female,Not Recorded In Database,Nonsmoker,NaN,NaN,Not Assessed,...,no,NaN,NaN,6/19/1996,Alive,NaN,NaN,2/23/1995,28,Not Collected
3,AMC-004,Stanford,80,Not Collected,Female,Not Recorded In Database,Nonsmoker,NaN,NaN,Not Assessed,...,no,NaN,NaN,12/13/1996,Alive,NaN,NaN,12/26/1992,47,Not Collected
4,AMC-005,Stanford,76,Not Collected,Male,Not Recorded In Database,Former,30,1962.0,Not Assessed,...,yes,distant,1/4/1996,1/7/1997,Alive,NaN,NaN,7/21/1994,2,Not Collected


In [25]:
# 看看原始临床表里有多少 R01 病例
raw_labels["Case ID"] = raw_labels["Case ID"].astype(str).str.strip()

r01_mask = raw_labels["Case ID"].str.startswith("R01-")

print("Total rows:", len(raw_labels))
print("R01 rows:", r01_mask.sum())

print("\nExample R01 IDs:")
print(raw_labels.loc[r01_mask, "Case ID"].head(20).tolist())

Total rows: 211
R01 rows: 162

Example R01 IDs:
['R01-001', 'R01-002', 'R01-003', 'R01-004', 'R01-005', 'R01-006', 'R01-007', 'R01-008', 'R01-009', 'R01-010', 'R01-011', 'R01-012', 'R01-013', 'R01-014', 'R01-015', 'R01-016', 'R01-017', 'R01-018', 'R01-019', 'R01-020']


In [26]:
clinical_cols = [
    "Case ID",
    "Age at Histological Diagnosis",
    "Gender",
    "Smoking status",
    "Pack Years",
    "Weight (lbs)",
]

clinical_raw = raw_labels[clinical_cols].copy()

clinical_97 = labels_df[["Case ID", "EGFR mutation status"]].merge(
    clinical_raw,
    on="Case ID",
    how="left",
    validate="one_to_one"
)

print("Merged clinical shape:", clinical_97.shape)

print("\nMissing values:")
print(clinical_97.isna().sum())

display(clinical_97.head())

Merged clinical shape: (97, 7)

Missing values:
Case ID                           0
EGFR mutation status              0
Age at Histological Diagnosis     0
Gender                            0
Smoking status                    0
Pack Years                       16
Weight (lbs)                      0
dtype: int64


,Case ID,EGFR mutation status,Age at Histological Diagnosis,Gender,Smoking status,Pack Years,Weight (lbs)
0,R01-003,Mutant,65,Male,Current,38,Not Collected
1,R01-005,Wildtype,84,Male,Former,20,145
2,R01-006,Wildtype,62,Male,Former,Not Collected,Not Collected
3,R01-008,Wildtype,73,Male,Nonsmoker,NaN,102
4,R01-012,Wildtype,70,Female,Nonsmoker,NaN,125


In [27]:
for col in [
    "Age at Histological Diagnosis",
    "Gender",
    "Smoking status",
    "Pack Years",
    "Weight (lbs)",
]:
    print("\n====", col, "====")
    print(clinical_97[col].value_counts(dropna=False).head(30))


==== Age at Histological Diagnosis ====
Age at Histological Diagnosis
65    7
63    7
70    6
68    6
62    5
73    5
74    5
81    5
67    5
78    4
64    3
71    3
66    3
69    3
76    3
84    2
59    2
86    2
72    2
77    2
80    2
79    2
43    1
87    1
56    1
53    1
75    1
85    1
47    1
51    1
Name: count, dtype: int64

==== Gender ====
Gender
Male      71
Female    26
Name: count, dtype: int64

==== Smoking status ====
Smoking status
Former       59
Current      22
Nonsmoker    16
Name: count, dtype: int64

==== Pack Years ====
Pack Years
NaN              16
50               10
40                7
30                7
Not Collected     5
10                5
90                5
20                4
60                4
80                3
25                3
8                 3
120               2
22                2
35                2
15                2
70                2
5                 2
38                1
42                1
34                1
43                

In [28]:
missing_tokens = {
    "Not Collected",
    "Not Assessed",
    "Not Recorded In Database",
    "Unknown",
    "",
}

clinical_check = clinical_97.copy()

for col in clinical_check.columns:
    clinical_check[col] = clinical_check[col].replace(
        list(missing_tokens),
        np.nan
    )

print("True missing values after cleaning:")
print(clinical_check.isna().sum())

True missing values after cleaning:
Case ID                           0
EGFR mutation status              0
Age at Histological Diagnosis     0
Gender                            0
Smoking status                    0
Pack Years                       21
Weight (lbs)                      5
dtype: int64


In [29]:
clinical_main_cols = [
    "Age at Histological Diagnosis",
    "Gender",
    "Smoking status",
]

clinical_model_df = clinical_check[
    ["Case ID", "EGFR mutation status"] + clinical_main_cols
].copy()

print("Clinical model shape:", clinical_model_df.shape)
print(clinical_model_df.isna().sum())

Clinical model shape: (97, 5)
Case ID                          0
EGFR mutation status             0
Age at Histological Diagnosis    0
Gender                           0
Smoking status                   0
dtype: int64


In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

clinical_numeric = [
    "Age at Histological Diagnosis"
]

clinical_categorical = [
    "Gender",
    "Smoking status"
]

clinical_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("scaler", StandardScaler())
        ]),
        clinical_numeric
    ),
    (
        "cat",
        Pipeline([
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        clinical_categorical
    ),
])

clinical_pipe = Pipeline([
    ("preprocessor", clinical_preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000,
        random_state=2026
    ))
])

clinical_param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

In [2]:
X_clinical = clinical_model_df[clinical_main_cols].copy()

y_clinical = (
    clinical_model_df["EGFR mutation status"]
    .map({
        "Wildtype": 0,
        "Mutant": 1
    })
    .astype(int)
)

clinical_repeat_results = []

N_REPEATS = 20

for repeat in range(N_REPEATS):

    seed = 2026 + repeat

    outer_cv_rep = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    inner_cv_rep = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )

    oof_pred_clinical = np.zeros(len(X_clinical), dtype=float)

    for train_idx, test_idx in outer_cv_rep.split(
        X_clinical, y_clinical
    ):

        X_train = X_clinical.iloc[train_idx]
        X_test = X_clinical.iloc[test_idx]

        y_train = y_clinical.iloc[train_idx]
        y_test = y_clinical.iloc[test_idx]

        grid_clinical = GridSearchCV(
            estimator=clinical_pipe,
            param_grid=clinical_param_grid,
            scoring="roc_auc",
            cv=inner_cv_rep,
            n_jobs=-1,
            refit=True
        )

        grid_clinical.fit(X_train, y_train)

        oof_pred_clinical[test_idx] = (
            grid_clinical.predict_proba(X_test)[:, 1]
        )

    auc = roc_auc_score(
        y_clinical,
        oof_pred_clinical
    )

    pr_auc = average_precision_score(
        y_clinical,
        oof_pred_clinical
    )

    clinical_repeat_results.append({
        "repeat": repeat + 1,
        "seed": seed,
        "roc_auc": auc,
        "pr_auc": pr_auc
    })

    print(
        f"Repeat {repeat + 1:02d}: "
        f"AUC={auc:.3f}, "
        f"PR-AUC={pr_auc:.3f}"
    )

clinical_repeat_df = pd.DataFrame(clinical_repeat_results)

print("\nClinical-only ROC-AUC:")
print(clinical_repeat_df["roc_auc"].describe())

print("\nClinical-only PR-AUC:")
print(clinical_repeat_df["pr_auc"].describe())

NameError: name 'clinical_model_df' is not defined

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"F:\yingxiang2")

LABEL_PATH = PROJECT_ROOT / "final_patients_egfr.csv"
RAW_LABEL_PATH = (
    PROJECT_ROOT
    / "NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv"
)

labels_df = pd.read_csv(LABEL_PATH)
raw_labels = pd.read_csv(RAW_LABEL_PATH)

raw_labels["Case ID"] = raw_labels["Case ID"].astype(str).str.strip()

clinical_cols = [
    "Case ID",
    "Age at Histological Diagnosis",
    "Gender",
    "Smoking status",
    "Pack Years",
    "Weight (lbs)",
]

clinical_raw = raw_labels[clinical_cols].copy()

clinical_97 = labels_df[
    ["Case ID", "EGFR mutation status"]
].merge(
    clinical_raw,
    on="Case ID",
    how="left",
    validate="one_to_one"
)

missing_tokens = {
    "Not Collected",
    "Not Assessed",
    "Not Recorded In Database",
    "Unknown",
    "",
}

clinical_check = clinical_97.copy()

for col in clinical_check.columns:
    clinical_check[col] = clinical_check[col].replace(
        list(missing_tokens),
        np.nan
    )

clinical_main_cols = [
    "Age at Histological Diagnosis",
    "Gender",
    "Smoking status",
]

clinical_model_df = clinical_check[
    ["Case ID", "EGFR mutation status"] + clinical_main_cols
].copy()

print("Clinical model shape:", clinical_model_df.shape)
print(clinical_model_df.isna().sum())

Clinical model shape: (97, 5)
Case ID                          0
EGFR mutation status             0
Age at Histological Diagnosis    0
Gender                           0
Smoking status                   0
dtype: int64


In [4]:
X_clinical = clinical_model_df[clinical_main_cols].copy()

y_clinical = (
    clinical_model_df["EGFR mutation status"]
    .map({
        "Wildtype": 0,
        "Mutant": 1
    })
    .astype(int)
)

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"F:\yingxiang2")

# =========================
# 1. 重新读取 radiomics
# =========================
FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "radiomics_features.csv"
)

features_df = pd.read_csv(FEATURE_PATH)

# =========================
# 2. 重新读取 EGFR 标签
# =========================
LABEL_PATH = PROJECT_ROOT / "final_patients_egfr.csv"

labels_df = pd.read_csv(LABEL_PATH)

# =========================
# 3. 重新读取原始临床表
# =========================
RAW_LABEL_PATH = (
    PROJECT_ROOT
    / "NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv"
)

raw_labels = pd.read_csv(RAW_LABEL_PATH)
raw_labels["Case ID"] = (
    raw_labels["Case ID"]
    .astype(str)
    .str.strip()
)

clinical_cols = [
    "Case ID",
    "Age at Histological Diagnosis",
    "Gender",
    "Smoking status",
    "Pack Years",
    "Weight (lbs)",
]

clinical_raw = raw_labels[clinical_cols].copy()

clinical_97 = labels_df[
    ["Case ID", "EGFR mutation status"]
].merge(
    clinical_raw,
    on="Case ID",
    how="left",
    validate="one_to_one"
)

# =========================
# 4. 清理特殊缺失值
# =========================
missing_tokens = [
    "Not Collected",
    "Not Assessed",
    "Not Recorded In Database",
    "Unknown",
    "",
]

clinical_check = clinical_97.replace(
    missing_tokens,
    np.nan
)

# 主 clinical baseline 变量
clinical_main_cols = [
    "Age at Histological Diagnosis",
    "Gender",
    "Smoking status",
]

clinical_model_df = clinical_check[
    ["Case ID", "EGFR mutation status"]
    + clinical_main_cols
].copy()

# =========================
# 5. 检查
# =========================
print("Radiomics:", features_df.shape)
print("Labels:", labels_df.shape)
print("Clinical:", clinical_model_df.shape)

print("\nClinical missing:")
print(clinical_model_df.isna().sum())

Radiomics: (97, 108)
Labels: (97, 2)
Clinical: (97, 5)

Clinical missing:
Case ID                          0
EGFR mutation status             0
Age at Histological Diagnosis    0
Gender                           0
Smoking status                   0
dtype: int64


In [6]:
combined_df = (
    features_df
    .merge(
        clinical_model_df[
            [
                "Case ID",
                "Age at Histological Diagnosis",
                "Gender",
                "Smoking status",
            ]
        ],
        on="Case ID",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        labels_df[
            ["Case ID", "EGFR mutation status"]
        ],
        on="Case ID",
        how="inner",
        validate="one_to_one"
    )
)

print("Combined shape:", combined_df.shape)
print("Duplicates:", combined_df["Case ID"].duplicated().sum())
print("Missing values:", combined_df.isna().sum().sum())

Combined shape: (97, 112)
Duplicates: 0
Missing values: 0


In [7]:
radiomics_cols = [
    c for c in combined_df.columns
    if c.startswith("original_")
]

clinical_numeric = [
    "Age at Histological Diagnosis"
]

clinical_categorical = [
    "Gender",
    "Smoking status"
]

combined_feature_cols = (
    radiomics_cols
    + clinical_numeric
    + clinical_categorical
)

X_combined = combined_df[combined_feature_cols].copy()

y_combined = (
    combined_df["EGFR mutation status"]
    .map({
        "Wildtype": 0,
        "Mutant": 1
    })
    .astype(int)
)

print("X_combined shape:", X_combined.shape)
print(y_combined.value_counts().sort_index())

X_combined shape: (97, 110)
EGFR mutation status
0    74
1    23
Name: count, dtype: int64


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

combined_preprocessor = ColumnTransformer([
    (
        "radiomics",
        StandardScaler(),
        radiomics_cols
    ),
    (
        "clinical_num",
        StandardScaler(),
        clinical_numeric
    ),
    (
        "clinical_cat",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ),
        clinical_categorical
    ),
])

combined_pipe = Pipeline([
    ("preprocessor", combined_preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000,
        random_state=2026
    ))
])

combined_param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

In [9]:
combined_repeat_results = []

N_REPEATS = 20

for repeat in range(N_REPEATS):

    seed = 2026 + repeat

    outer_cv_rep = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    inner_cv_rep = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )

    oof_pred_combined = np.zeros(len(X_combined), dtype=float)

    for train_idx, test_idx in outer_cv_rep.split(
        X_combined,
        y_combined
    ):

        X_train = X_combined.iloc[train_idx]
        X_test = X_combined.iloc[test_idx]

        y_train = y_combined.iloc[train_idx]
        y_test = y_combined.iloc[test_idx]

        grid_combined = GridSearchCV(
            estimator=combined_pipe,
            param_grid=combined_param_grid,
            scoring="roc_auc",
            cv=inner_cv_rep,
            n_jobs=-1,
            refit=True
        )

        grid_combined.fit(X_train, y_train)

        oof_pred_combined[test_idx] = (
            grid_combined.predict_proba(X_test)[:, 1]
        )

    auc = roc_auc_score(
        y_combined,
        oof_pred_combined
    )

    pr_auc = average_precision_score(
        y_combined,
        oof_pred_combined
    )

    combined_repeat_results.append({
        "repeat": repeat + 1,
        "seed": seed,
        "roc_auc": auc,
        "pr_auc": pr_auc
    })

    print(
        f"Repeat {repeat + 1:02d}: "
        f"AUC={auc:.3f}, "
        f"PR-AUC={pr_auc:.3f}"
    )

combined_repeat_df = pd.DataFrame(combined_repeat_results)

print("\nCombined ROC-AUC:")
print(combined_repeat_df["roc_auc"].describe())

print("\nCombined PR-AUC:")
print(combined_repeat_df["pr_auc"].describe())

Repeat 01: AUC=0.463, PR-AUC=0.248
Repeat 02: AUC=0.469, PR-AUC=0.237
Repeat 03: AUC=0.608, PR-AUC=0.330
Repeat 04: AUC=0.559, PR-AUC=0.280
Repeat 05: AUC=0.589, PR-AUC=0.322
Repeat 06: AUC=0.550, PR-AUC=0.290
Repeat 07: AUC=0.640, PR-AUC=0.346
Repeat 08: AUC=0.600, PR-AUC=0.304
Repeat 09: AUC=0.598, PR-AUC=0.361
Repeat 10: AUC=0.579, PR-AUC=0.332
Repeat 11: AUC=0.595, PR-AUC=0.337
Repeat 12: AUC=0.580, PR-AUC=0.355
Repeat 13: AUC=0.660, PR-AUC=0.337
Repeat 14: AUC=0.572, PR-AUC=0.328
Repeat 15: AUC=0.603, PR-AUC=0.376
Repeat 16: AUC=0.578, PR-AUC=0.301
Repeat 17: AUC=0.592, PR-AUC=0.305
Repeat 18: AUC=0.586, PR-AUC=0.291
Repeat 19: AUC=0.538, PR-AUC=0.298
Repeat 20: AUC=0.682, PR-AUC=0.406

Combined ROC-AUC:
count    20.000000
mean      0.582080
std       0.052349
min       0.462985
25%       0.568596
50%       0.587544
75%       0.601058
max       0.681551
Name: roc_auc, dtype: float64

Combined PR-AUC:
count    20.000000
mean      0.319274
std       0.040859
min       0.236838
25%  

In [10]:
combined_repeat_df.to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_combined.csv",
    index=False
)

print("Saved combined results.")

Saved combined results.


In [11]:
radiomics_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_l2_logistic.csv"
)

clinical_repeat_df = pd.DataFrame(clinical_repeat_results)

combined_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_combined.csv"
)

NameError: name 'clinical_repeat_results' is not defined

In [12]:
radiomics_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_l2_logistic.csv"
)

clinical_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_clinical_only.csv"
)

combined_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_combined.csv"
)

print("Radiomics:", radiomics_repeat_df.shape)
print("Clinical:", clinical_repeat_df.shape)
print("Combined:", combined_repeat_df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'F:\\yingxiang2\\results\\metrics\\repeated_nested_cv_clinical_only.csv'

In [13]:
clinical_repeat_df = pd.DataFrame({
    "repeat": list(range(1, 21)),
    "seed": list(range(2026, 2046)),
    "roc_auc": [
        0.456, 0.548, 0.632, 0.509, 0.611,
        0.591, 0.575, 0.541, 0.602, 0.535,
        0.591, 0.596, 0.624, 0.507, 0.614,
        0.582, 0.617, 0.522, 0.595, 0.572
    ],
    "pr_auc": [
        0.229, 0.270, 0.327, 0.248, 0.305,
        0.314, 0.324, 0.256, 0.327, 0.252,
        0.294, 0.312, 0.317, 0.236, 0.300,
        0.287, 0.322, 0.270, 0.321, 0.307
    ]
})

CLINICAL_OUTPUT = (
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_clinical_only.csv"
)

clinical_repeat_df.to_csv(
    CLINICAL_OUTPUT,
    index=False
)

print("Saved:", CLINICAL_OUTPUT)
print(clinical_repeat_df.shape)
print(clinical_repeat_df.head())

Saved: F:\yingxiang2\results\metrics\repeated_nested_cv_clinical_only.csv
(20, 4)
   repeat  seed  roc_auc  pr_auc
0       1  2026    0.456   0.229
1       2  2027    0.548   0.270
2       3  2028    0.632   0.327
3       4  2029    0.509   0.248
4       5  2030    0.611   0.305


In [14]:
radiomics_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_l2_logistic.csv"
)

clinical_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_clinical_only.csv"
)

combined_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_combined.csv"
)

print("Radiomics:", radiomics_repeat_df.shape)
print("Clinical:", clinical_repeat_df.shape)
print("Combined:", combined_repeat_df.shape)

Radiomics: (20, 4)
Clinical: (20, 4)
Combined: (20, 4)


In [15]:
model_compare = (
    radiomics_repeat_df
    .merge(
        clinical_repeat_df,
        on=["repeat", "seed"],
        suffixes=("_radiomics", "_clinical")
    )
    .merge(
        combined_repeat_df,
        on=["repeat", "seed"]
    )
    .rename(columns={
        "roc_auc": "roc_auc_combined",
        "pr_auc": "pr_auc_combined"
    })
)

model_compare["auc_rad_minus_clin"] = (
    model_compare["roc_auc_radiomics"]
    - model_compare["roc_auc_clinical"]
)

model_compare["auc_rad_minus_combined"] = (
    model_compare["roc_auc_radiomics"]
    - model_compare["roc_auc_combined"]
)

model_compare["pr_rad_minus_clin"] = (
    model_compare["pr_auc_radiomics"]
    - model_compare["pr_auc_clinical"]
)

model_compare["pr_rad_minus_combined"] = (
    model_compare["pr_auc_radiomics"]
    - model_compare["pr_auc_combined"]
)

print("Radiomics vs Clinical — AUC difference:")
print(model_compare["auc_rad_minus_clin"].describe())

print(
    "\nRadiomics better than Clinical:",
    (model_compare["auc_rad_minus_clin"] > 0).sum(),
    "of 20"
)

print("\nRadiomics vs Combined — AUC difference:")
print(model_compare["auc_rad_minus_combined"].describe())

print(
    "\nRadiomics better than Combined:",
    (model_compare["auc_rad_minus_combined"] > 0).sum(),
    "of 20"
)

print("\nRadiomics vs Clinical — PR-AUC difference:")
print(model_compare["pr_rad_minus_clin"].describe())

print("\nRadiomics vs Combined — PR-AUC difference:")
print(model_compare["pr_rad_minus_combined"].describe())

Radiomics vs Clinical — AUC difference:
count    20.000000
mean      0.056850
std       0.052913
min      -0.048257
25%       0.010649
50%       0.056341
75%       0.098139
max       0.144664
Name: auc_rad_minus_clin, dtype: float64

Radiomics better than Clinical: 17 of 20

Radiomics vs Combined — AUC difference:
count    20.000000
mean      0.045770
std       0.049875
min      -0.068743
25%       0.012338
50%       0.056992
75%       0.077262
max       0.130435
Name: auc_rad_minus_combined, dtype: float64

Radiomics better than Combined: 16 of 20

Radiomics vs Clinical — PR-AUC difference:
count    20.000000
mean      0.061672
std       0.052905
min      -0.045450
25%       0.015674
50%       0.070099
75%       0.095731
max       0.145535
Name: pr_rad_minus_clin, dtype: float64

Radiomics vs Combined — PR-AUC difference:
count    20.000000
mean      0.033297
std       0.045300
min      -0.030600
25%      -0.006267
50%       0.033229
75%       0.068833
max       0.115979
Name: pr_rad_

In [16]:
print("clinical_model_df:", "clinical_model_df" in globals())
print("clinical_pipe:", "clinical_pipe" in globals())
print("clinical_param_grid:", "clinical_param_grid" in globals())

clinical_model_df: True
clinical_pipe: False
clinical_param_grid: False


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

clinical_numeric = [
    "Age at Histological Diagnosis"
]

clinical_categorical = [
    "Gender",
    "Smoking status"
]

clinical_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("scaler", StandardScaler())
        ]),
        clinical_numeric
    ),
    (
        "cat",
        Pipeline([
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        clinical_categorical
    ),
])

clinical_pipe = Pipeline([
    ("preprocessor", clinical_preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000,
        random_state=2026
    ))
])

clinical_param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

print("clinical_pipe restored:", clinical_pipe is not None)
print("clinical_param_grid restored:", clinical_param_grid)

clinical_pipe restored: True
clinical_param_grid restored: {'model__C': [0.001, 0.01, 0.1, 1, 10, 100]}


In [18]:
X_clinical = clinical_model_df[
    [
        "Age at Histological Diagnosis",
        "Gender",
        "Smoking status",
    ]
].copy()

y_clinical = (
    clinical_model_df["EGFR mutation status"]
    .map({
        "Wildtype": 0,
        "Mutant": 1
    })
    .astype(int)
)

clinical_repeat_results = []

for repeat in range(20):

    seed = 2026 + repeat

    outer_cv_rep = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    inner_cv_rep = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=seed
    )

    oof_pred = np.zeros(len(X_clinical), dtype=float)

    for train_idx, test_idx in outer_cv_rep.split(
        X_clinical,
        y_clinical
    ):

        grid = GridSearchCV(
            estimator=clinical_pipe,
            param_grid=clinical_param_grid,
            scoring="roc_auc",
            cv=inner_cv_rep,
            n_jobs=-1,
            refit=True
        )

        grid.fit(
            X_clinical.iloc[train_idx],
            y_clinical.iloc[train_idx]
        )

        oof_pred[test_idx] = grid.predict_proba(
            X_clinical.iloc[test_idx]
        )[:, 1]

    auc = roc_auc_score(
        y_clinical,
        oof_pred
    )

    pr_auc = average_precision_score(
        y_clinical,
        oof_pred
    )

    clinical_repeat_results.append({
        "repeat": repeat + 1,
        "seed": seed,
        "roc_auc": auc,
        "pr_auc": pr_auc
    })

    print(
        f"Repeat {repeat + 1:02d}: "
        f"AUC={auc:.6f}, "
        f"PR-AUC={pr_auc:.6f}"
    )

clinical_repeat_df = pd.DataFrame(
    clinical_repeat_results
)

clinical_repeat_df.to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_clinical_only.csv",
    index=False
)

print("\nClinical ROC-AUC:")
print(clinical_repeat_df["roc_auc"].describe())

print("\nClinical PR-AUC:")
print(clinical_repeat_df["pr_auc"].describe())

print("\nSaved exact clinical results.")

Repeat 01: AUC=0.456228, PR-AUC=0.229293
Repeat 02: AUC=0.548472, PR-AUC=0.269913
Repeat 03: AUC=0.631610, PR-AUC=0.327169
Repeat 04: AUC=0.509401, PR-AUC=0.247686
Repeat 05: AUC=0.611340, PR-AUC=0.304659
Repeat 06: AUC=0.591069, PR-AUC=0.313981
Repeat 07: AUC=0.575499, PR-AUC=0.323888
Repeat 08: AUC=0.540834, PR-AUC=0.255932
Repeat 09: AUC=0.601645, PR-AUC=0.327232
Repeat 10: AUC=0.534959, PR-AUC=0.251955
Repeat 11: AUC=0.590776, PR-AUC=0.293591
Repeat 12: AUC=0.595770, PR-AUC=0.311771
Repeat 13: AUC=0.623972, PR-AUC=0.317279
Repeat 14: AUC=0.507051, PR-AUC=0.236459
Repeat 15: AUC=0.613690, PR-AUC=0.300493
Repeat 16: AUC=0.581962, PR-AUC=0.286925
Repeat 17: AUC=0.616627, PR-AUC=0.321995
Repeat 18: AUC=0.521739, PR-AUC=0.269838
Repeat 19: AUC=0.595182, PR-AUC=0.321165
Repeat 20: AUC=0.571680, PR-AUC=0.306808

Clinical ROC-AUC:
count    20.000000
mean      0.570975
std       0.046613
min       0.456228
25%       0.539365
50%       0.586369
75%       0.604069
max       0.631610
Name: roc

In [19]:
radiomics_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_l2_logistic.csv"
)

clinical_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_clinical_only.csv"
)

combined_repeat_df = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "repeated_nested_cv_combined.csv"
)

model_compare = (
    radiomics_repeat_df
    .merge(
        clinical_repeat_df,
        on=["repeat", "seed"],
        suffixes=("_radiomics", "_clinical")
    )
    .merge(
        combined_repeat_df,
        on=["repeat", "seed"]
    )
    .rename(columns={
        "roc_auc": "roc_auc_combined",
        "pr_auc": "pr_auc_combined"
    })
)

model_compare["auc_rad_minus_clin"] = (
    model_compare["roc_auc_radiomics"]
    - model_compare["roc_auc_clinical"]
)

model_compare["auc_rad_minus_combined"] = (
    model_compare["roc_auc_radiomics"]
    - model_compare["roc_auc_combined"]
)

model_compare["pr_rad_minus_clin"] = (
    model_compare["pr_auc_radiomics"]
    - model_compare["pr_auc_clinical"]
)

model_compare["pr_rad_minus_combined"] = (
    model_compare["pr_auc_radiomics"]
    - model_compare["pr_auc_combined"]
)

print("Radiomics vs Clinical — AUC difference:")
print(model_compare["auc_rad_minus_clin"].describe())

print(
    "\nRadiomics better than Clinical:",
    (model_compare["auc_rad_minus_clin"] > 0).sum(),
    "of 20"
)

print("\nRadiomics vs Combined — AUC difference:")
print(model_compare["auc_rad_minus_combined"].describe())

print(
    "\nRadiomics better than Combined:",
    (model_compare["auc_rad_minus_combined"] > 0).sum(),
    "of 20"
)

print("\nRadiomics vs Clinical — PR-AUC difference:")
print(model_compare["pr_rad_minus_clin"].describe())

print("\nRadiomics vs Combined — PR-AUC difference:")
print(model_compare["pr_rad_minus_combined"].describe())

model_compare.to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "model_comparison_repeated_nested_cv.csv",
    index=False
)

Radiomics vs Clinical — AUC difference:
count    20.000000
mean      0.056874
std       0.052851
min      -0.047885
25%       0.010870
50%       0.056110
75%       0.097826
max       0.144830
Name: auc_rad_minus_clin, dtype: float64

Radiomics better than Clinical: 17 of 20

Radiomics vs Combined — AUC difference:
count    20.000000
mean      0.045770
std       0.049875
min      -0.068743
25%       0.012338
50%       0.056992
75%       0.077262
max       0.130435
Name: auc_rad_minus_combined, dtype: float64

Radiomics better than Combined: 16 of 20

Radiomics vs Clinical — PR-AUC difference:
count    20.000000
mean      0.061670
std       0.052875
min      -0.045444
25%       0.015798
50%       0.069949
75%       0.096006
max       0.145580
Name: pr_rad_minus_clin, dtype: float64

Radiomics vs Combined — PR-AUC difference:
count    20.000000
mean      0.033297
std       0.045300
min      -0.030600
25%      -0.006267
50%       0.033229
75%       0.068833
max       0.115979
Name: pr_rad_